# 第14章 アレーアンテナとビーム制御

書籍『Sionna RTによる電波伝搬レイトレーシング入門』第14章のコードである。
セルは本文の掲載順に並べてあり、コードセルの中身は本文に印刷したものと同じである。

先頭の「準備」セルだけは本文に印刷していない。本文のコードが前提にしている
import とシーンの読み込みをここで済ませてある。準備セルを実行すれば、以降は
上から順に実行できる。

ただし、本文が説明のために示している断片（自分で作ったシーンのパスや、その
シーンには無い物体名を使う例）はそのままでは動かない。該当するセルの前に注記を
置いてある。

## 準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# 本書の多くの章で使うミュンヘン中心部の都市シーン
scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = PathSolver()
rmsolver = RadioMapSolver()

## 2. 平面アレーの設定

In [ ]:
scene.tx_array = PlanarArray(num_rows=4, num_cols=4,
                              vertical_spacing=0.5,
                              horizontal_spacing=0.5,
                              pattern='tr38901',
                              polarization='V')

## 5. ビーム方向を変える

In [ ]:
import numpy as np
import mitsuba as mi

# ローカル座標で方位角 phi、仰角 theta へ向ける例
phi, theta = np.deg2rad(30.0), np.deg2rad(0.0)
direction = np.array([np.cos(theta)*np.cos(phi),
                      np.cos(theta)*np.sin(phi),
                      np.sin(theta)])
pos = np.array(scene.tx_array.normalized_positions)  # (3, N), 波長単位
phase = 2*np.pi * (pos.T @ direction)
w = np.exp(1j*phase) / np.sqrt(phase.size)
precoding_vec = (mi.TensorXf(w.real), mi.TensorXf(w.imag))
rm = rmsolver(scene, max_depth=3, cell_size=[2.0, 2.0],
              precoding_vec=precoding_vec)